In [1]:
!pip install openai
!pip install langchain
!pip install tqdm
!pip install chromadb
!pip install tiktoken
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

In [2]:
# key 설정

import os
import openai

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('kosa')

openai.api_key = os.getenv("OPENAI_API_KEY")

In [3]:
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/hwchase17/chat-your-data/master/state_of_the_union.txt",
                           filename="state_of_the_union.txt")


('state_of_the_union.txt', <http.client.HTTPMessage at 0x7c0c90c3a690>)

In [4]:
from tqdm import tqdm

Simple TextLoader 구현해 보기

In [5]:
class SimpleTextLoader:
    # 생성자 (준비물)
    def __init__(self, file_path):
        self.file_path = file_path

    def load(self):
      text =''      # 파일에서 읽은 데이터 저장하는 변수
      with open(self.file_path, 'r', encoding='utf-8') as f:
         text = f.read()
      return text

In [6]:
# class SimpleCharacterTextSplitter:
#     def __init__(self, chunk_size, chunk_overlap):
#         self.chunk_size = chunk_size
#         self.chunk_overlap = chunk_overlap

#     def split_document(self, text):
#         chunks = []
#         start = 0
#         while start < len(text):
#             end = start + self.chunk_size
#             chunks.append(text[start:end])
#             start += self.chunk_size - self.chunk_overlap
#         return chunks

In [7]:
# # 이삿짐 센터
# # 여러 가지 box size
# # box 잇사 전에 가져다 줌
# # 짐을 박스에 담겠죠

class SimpleCharacterTextSplitter:

  def __init__(self, chunk_size, chunk_overlap, seperator_pattern="\n\n"):
    self.chunk_size = chunk_size
    self.chunk_overlap = chunk_overlap
    self.seperator_pattern = seperator_pattern

  # 문장 분할 함수(메소드)
  def split_document(self, documents):

    # 파일 전체 내용 >> 문단 단위로 나뉘기 (기준: seperator_pattern)
    splits = documents.split(self.seperator_pattern)

    chunks = [] # 최종적으로 생성될 chunks 저장
    current_chunk = splits[0]
    # 첫번째 문단 >> 초기 chunk 설정

    for split in tqdm(splits[1:], desc="splitting...."):
      # current chunk(현재 문단, 즉 첫번째 문단 >> splits[0])
      # 다음 문단, 그리고 구분자가 chunk_size 초과되는지 확인
      if len(current_chunk) + len(split) + len(self.seperator_pattern) > self.chunk_size:
        chunks.append(current_chunk.strip())
        current_chunk = split
      else:
        # 초과하지 않으면 >> 현재 청크에 구분자 + 다음 문단 추가
        current_chunk += split + self.seperator_pattern
        # current_chunk = current_chunk + split + separator_pattern

    # 마지막 청크 추가
    if current_chunk:
      chunks.append(current_chunk.strip())

    return chunks

#         # A split 500자
#         # B split 10자
#         # C split 600자

#         # 위 split들을 청크(박스)에 담는 행위
#         # A + B = 510 첫번째 박스
#         # C는 600자라서 A + B + C = 1110 > 1000 박스 크기 초과
#         # C 두번째 박스
#         # 첫번째 박스 (510 : A, B)
#         # 두번째 박스 (600 : C)

SimpleOpenAIEmbeddings 구현해보기

In [8]:
from openai import OpenAI

class SimpleOpenAIEmbeddings:

  def embed_query(self, text):
    client = OpenAI()
    response = client.embeddings.create(
                input = text,
                model = 'text-embedding-ada-002'
    )
    return response.data[0].embedding

SimpleVectorStore 구현

In [9]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class SimpleVectorStore:

  def __init__(self, docs, embedding):
    self.embedding = embedding
    self.documents = []   # 문서 내용을 저장할 리스트
    self.vectors = []     # 문서의 벡터 저장할 리스트

    #모든 문서에 대해 임베딩 수행 >> 벡터, 문서내용 저장
    for doc in tqdm(docs, desc="embedding..."):
      self.documents.append(doc)
      vector = self.embedding.embed_query(doc) # 문서 >> 벡터(숫자) 변환
      self.vectors.append(vector)

  # 유사도 검색 메소드
  def similarity_search(self, query, k=4): # 가장 유사한 4개 문서를 찾아줘.
      query_vector = self.embedding.embed_query(query) # query >> vector

      if not self.vectors:   # 저장된 벡터가 없으면? >> 빈 리스트 반환
        return []

      similarities = cosine_similarity([query_vector], self.vectors)[0]
      # query vector - 저장되어 있는 모든 vector 간 코사인 유사도 계산
      sorted_doc_similarities = sorted(zip(self.documents, similarities), key=lambda x: x[1], reverse=True)
      # zip 함수 사용, (문서, 유사도) 튜플 형태로 묶어줌, 유사도 기준, 내림차순 정렬
      return sorted_doc_similarities[:k] # 상위 k 개 문서 반환

  def as_retriever(self, k=4):
    return SimpleRetriever(self, k)

SimpleRetriever 구현

In [11]:
class SimpleRetriever:
  def __init__(self, vector_store, k=4):
    self.vector_store = vector_store
    self.k = k

  def get_relevant_documents(self, query):
    docs = self.vector_store.similarity_search(query, self.k)
    return docs

In [12]:
raw_documents = SimpleTextLoader('state_of_the_union.txt').load()
text_splitter = SimpleCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_document(raw_documents)
print(raw_documents)
print(documents)
print(len(documents))

splitting....: 100%|██████████| 358/358 [00:00<00:00, 410711.39it/s]

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world. 

Groups of citizens blocking tanks with their bodies. Every

In [13]:
text_splitter = SimpleCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_document(raw_documents)
print(documents)

splitting....: 100%|██████████| 358/358 [00:00<00:00, 589540.96it/s]

['Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  Last year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.', 'Groups of citizens blocking tanks with thei

In [14]:
len(documents)

42

In [15]:
documents[0]

'Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  Last year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.'

In [16]:
# 문자 >> 숫자(벡터)
db = SimpleVectorStore(documents, SimpleOpenAIEmbeddings())

embedding...: 100%|██████████| 42/42 [00:14<00:00,  2.81it/s]


In [17]:
len(documents)

42

In [18]:
documents[0:4]

['Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  Last year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.',
 'Groups of citizens blocking tanks with the

In [19]:
query = "What did the president say about Ketanji Brown Jackson"

In [20]:
docs = db.similarity_search(query)
docs

[('Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.',
  np.float64(0.8127522810662635)),
 ('A former top litigator in private practice. A former federal public defender. And from a family of public school educators and police officers. A consensus builder. Si

In [21]:
print(docs[0])

('Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.', np.float64(0.8127522810662635))


In [22]:
docs[0]

('Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.',
 np.float64(0.8127522810662635))

In [23]:
docs[1]

('A former top litigator in private practice. A former federal public defender. And from a family of public school educators and police officers. A consensus builder. Since she’s been nominated, she’s received a broad range of support—from the Fraternal Order of Police to former judges appointed by Democrats and Republicans. And if we are to advance liberty and justice, we need to secure the Border and fix the immigration system. \n\nWe can do both. At our border, we’ve installed new technology like cutting-edge scanners to better detect drug smuggling.  \n\nWe’ve set up joint patrols with Mexico and Guatemala to catch more human traffickers.  \n\nWe’re putting in place dedicated immigration judges so families fleeing persecution and violence can have their cases heard faster. \n\nWe’re securing commitments and supporting partners in South and Central America to host more refugees and secure their own borders.',
 np.float64(0.7823524965908014))

In [24]:
print(docs[0][0])

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.


In [25]:
print(docs[0][1])

0.8127522810662635


한글 벡터스토어

In [26]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/puzzlet/constitution-kr/master/%EB%8C%80%ED%95%9C%EB%AF%BC%EA%B5%AD%20%ED%97%8C%EB%B2%95.txt",
    filename="korea_constitution.txt"
)

('korea_constitution.txt', <http.client.HTTPMessage at 0x7c0c667f9be0>)

In [27]:
raw_documents = SimpleTextLoader('korea_constitution.txt').load()
text_splitter = SimpleCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_document(raw_documents)

splitting....: 100%|██████████| 152/152 [00:00<00:00, 189573.06it/s]


In [28]:
documents

['전문유구한 역사와 전통에 빛나는 우리 대한국민은 3·1운동으로 건립된 대한민국임시정부의 법통과 불의에 항거한 4·19민주이념을 계승하고, 조국의 민주개혁과 평화적 통일의 사명에 입각하여 정의·인도와 동포애로써 민족의 단결을 공고히 하고, 모든 사회적 폐습과 불의를 타파하며, 자율과 조화를 바탕으로 자유민주적 기본질서를 더욱 확고히 하여 정치·경제·사회·문화의 모든 영역에 있어서 각인의 기회를 균등히 하고, 능력을 최고도로 발휘하게 하며, 자유와 권리에 따르는 책임과 의무를 완수하게 하여, 안으로는 국민생활의 균등한 향상을 기하고 밖으로는 항구적인 세계평화와 인류공영에 이바지함으로써 우리들과 우리들의 자손의 안전과 자유와 행복을 영원히 확보할 것을 다짐하면서 1948년 7월 12일에 제정되고 8차에 걸쳐 개정된 헌법을 이제 국회의 의결을 거쳐 국민투표에 의하여 개정한다.\n\n제1장 총강\n\n제1조 ① 대한민국은 민주공화국이다.\n② 대한민국의 주권은 국민에게 있고, 모든 권력은 국민으로부터 나온다.\n\n제2조 ① 대한민국의 국민이 되는 요건은 법률로 정한다.\n② 국가는 법률이 정하는 바에 의하여 재외국민을 보호할 의무를 진다.\n\n제3조 대한민국의 영토는 한반도와 그 부속도서로 한다.\n\n제4조 대한민국은 통일을 지향하며, 자유민주적 기본질서에 입각한 평화적 통일 정책을 수립하고 이를 추진한다.\n\n제5조 ① 대한민국은 국제평화의 유지에 노력하고 침략적 전쟁을 부인한다.\n② 국군은 국가의 안전보장과 국토방위의 신성한 의무를 수행함을 사명으로 하며, 그 정치적 중립성은 준수된다.\n\n제6조 ① 헌법에 의하여 체결·공포된 조약과 일반적으로 승인된 국제법규는 국내법과 같은 효력을 가진다.\n② 외국인은 국제법과 조약이 정하는 바에 의하여 그 지위가 보장된다.\n\n제7조 ① 공무원은 국민전체에 대한 봉사자이며, 국민에 대하여 책임을 진다.\n② 공무원의 신분과 정치적 중립성은 법률이 정하는 바에 의하여 보장된다.',
 '제8조 ① 정당의 설립은 자

In [29]:
db = SimpleVectorStore(documents, SimpleOpenAIEmbeddings())

embedding...: 100%|██████████| 22/22 [00:05<00:00,  3.87it/s]


In [30]:
query = "대한민국의 대통령 임기는?"
docs = db.similarity_search(query)
docs

[('제65조 ① 대통령·국무총리·국무위원·행정각부의 장·헌법재판소 재판관·법관·중앙선거관리위원회 위원·감사원장·감사위원 기타 법률이 정한 공무원이 그 직무집행에 있어서 헌법이나 법률을 위배한 때에는 국회는 탄핵의 소추를 의결할 수 있다.\n② 제1항의 탄핵소추는 국회재적의원 3분의 1 이상의 발의가 있어야 하며, 그 의결은 국회재적의원 과반수의 찬성이 있어야 한다. 다만, 대통령에 대한 탄핵소추는 국회재적의원 과반수의 발의와 국회재적의원 3분의 2 이상의 찬성이 있어야 한다.\n③ 탄핵소추의 의결을 받은 자는 탄핵심판이 있을 때까지 그 권한행사가 정지된다.\n④ 탄핵결정은 공직으로부터 파면함에 그친다. 그러나, 이에 의하여 민사상이나 형사상의 책임이 면제되지는 아니한다.제4장 정부\n제1절 대통령\n\n제66조 ① 대통령은 국가의 원수이며, 외국에 대하여 국가를 대표한다.\n② 대통령은 국가의 독립·영토의 보전·국가의 계속성과 헌법을 수호할 책무를 진다.\n③ 대통령은 조국의 평화적 통일을 위한 성실한 의무를 진다.\n④ 행정권은 대통령을 수반으로 하는 정부에 속한다.\n\n제67조 ① 대통령은 국민의 보통·평등·직접·비밀선거에 의하여 선출한다.\n② 제1항의 선거에 있어서 최고득표자가 2인 이상인 때에는 국회의 재적의원 과반수가 출석한 공개회의에서 다수표를 얻은 자를 당선자로 한다.\n③ 대통령후보자가 1인일 때에는 그 득표수가 선거권자 총수의 3분의 1 이상이 아니면 대통령으로 당선될 수 없다.\n④ 대통령으로 선거될 수 있는 자는 국회의원의 피선거권이 있고 선거일 현재 40세에 달하여야 한다.\n⑤ 대통령의 선거에 관한 사항은 법률로 정한다.\n\n제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.',
  np.float64(0.8517120897267707)),
 ('제76

벡터스토어 튜닝하기

In [31]:
raw_documents = SimpleTextLoader('korea_constitution.txt').load()
text_splitter = SimpleCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
documents = text_splitter.split_document(raw_documents)

splitting....: 100%|██████████| 152/152 [00:00<00:00, 468775.15it/s]


In [32]:
documents[:5]

['전문',
 '유구한 역사와 전통에 빛나는 우리 대한국민은 3·1운동으로 건립된 대한민국임시정부의 법통과 불의에 항거한 4·19민주이념을 계승하고, 조국의 민주개혁과 평화적 통일의 사명에 입각하여 정의·인도와 동포애로써 민족의 단결을 공고히 하고, 모든 사회적 폐습과 불의를 타파하며, 자율과 조화를 바탕으로 자유민주적 기본질서를 더욱 확고히 하여 정치·경제·사회·문화의 모든 영역에 있어서 각인의 기회를 균등히 하고, 능력을 최고도로 발휘하게 하며, 자유와 권리에 따르는 책임과 의무를 완수하게 하여, 안으로는 국민생활의 균등한 향상을 기하고 밖으로는 항구적인 세계평화와 인류공영에 이바지함으로써 우리들과 우리들의 자손의 안전과 자유와 행복을 영원히 확보할 것을 다짐하면서 1948년 7월 12일에 제정되고 8차에 걸쳐 개정된 헌법을 이제 국회의 의결을 거쳐 국민투표에 의하여 개정한다.',
 '제1장 총강제1조 ① 대한민국은 민주공화국이다.\n② 대한민국의 주권은 국민에게 있고, 모든 권력은 국민으로부터 나온다.',
 '제2조 ① 대한민국의 국민이 되는 요건은 법률로 정한다.\n② 국가는 법률이 정하는 바에 의하여 재외국민을 보호할 의무를 진다.',
 '제3조 대한민국의 영토는 한반도와 그 부속도서로 한다.제4조 대한민국은 통일을 지향하며, 자유민주적 기본질서에 입각한 평화적 통일 정책을 수립하고 이를 추진한다.']

In [33]:
db = SimpleVectorStore(documents, SimpleOpenAIEmbeddings())

embedding...: 100%|██████████| 137/137 [00:33<00:00,  4.10it/s]


In [34]:
query = "대한민국의 대통령 임기는?"
docs = db.similarity_search(query)
docs

[('제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.\n"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."',
  np.float64(0.8645506492340105)),
 ('제4장 정부\n제1절 대통령', np.float64(0.8574693499413382)),
 ('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.',
  np.float64(0.8562724466152125)),
 ('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', np.float64(0.8543761942597324))]

검색기

In [35]:
retriever = db.as_retriever()

In [36]:
query = "대한민국의 대통령 임기는?"

In [37]:
unique_docs = retriever.get_relevant_documents(query)
unique_docs

[('제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.\n"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."',
  np.float64(0.8645506492340105)),
 ('제4장 정부\n제1절 대통령', np.float64(0.8574693499413382)),
 ('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.',
  np.float64(0.8562724466152125)),
 ('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', np.float64(0.8543761942597324))]

챗봇 만들기

In [38]:
import openai

system_prompt_template = ("You are a helpful assistant. "
                          "Based on the following content, "
                          "kindly and comprehensively respond to user questions. write in Korean."
                          "[Content]"
                          "{content}"
                          "")

# 검색 기반 QA 클래스 정의
class SimpleRetrievalQA:
  def __init__(self, retriever):
    self.retriever = retriever
    # 검색기를 인스턴스 변수로 저장
  def invoke(self, query):
    docs = self.retriever.get_relevant_documents(query)
    print(docs)
    # query(질의)에 맞는 문서 검색

    # 검색한 문서 출력
    for i, doc in enumerate(docs):
      print("[#" + str(i)+ "]", doc[1]) # 문서의 내용
      print(doc[0])

    completion = openai.chat.completions.create(
        model = 'gpt-4o',
        messages = [
            {"role":"system", "content": system_prompt_template.format(content=docs)},
            {"role":"user", "content": query}]
    )
    # 시스템 메시지: 검색된 문서 내용 >> 시스템 프롬프트에 전달
    return completion.choices[0].message.content

In [39]:
chain = SimpleRetrievalQA(retriever)

answer = chain.invoke("대통령의 임기는?")
print('>>', answer)

[('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.', np.float64(0.8631684903584571)), ('제4장 정부\n제1절 대통령', np.float64(0.8561061351900602)), ('제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.\n"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."', np.float64(0.8561054073789893)), ('제3관 행정각부제94조 행정각부의 장은 국무위원 중에서 국무총리의 제청으로 대통령이 임명한다.', np.float64(0.8540742323730343))]
[#0] 0.8631684903584571
제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.
② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.
[#1] 0.8561061351900602
제4장 정부
제1절 대통령
[#2] 0.8561054073789893
제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.
"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."
[#3] 0.8540742323730343
제3관 행정각부제94조 행정각부의 장은 국무위원 중에서 국무총리의 제청으로 대통령이 임명한다.
>> 대통령의 임기는 5년입니다. 대통령의 임기가 만료될 때, 즉 임기 만료 70일 내지 40일 전에 후임자를 선거해야 합니다.


In [40]:
chain = SimpleRetrievalQA(retriever)

answer = chain.invoke("대통령은 중임을 할 수 있나?")
print('>>', answer)

[('제72조 대통령은 필요하다고 인정할 때에는 외교·국방·통일 기타 국가안위에 관한 중요정책을 국민투표에 붙일 수 있다.', np.float64(0.8698584460847605)), ('제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.\n"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."', np.float64(0.8661055841490819)), ('제76조 ① 대통령은 내우·외환·천재·지변 또는 중대한 재정·경제상의 위기에 있어서 국가의 안전보장 또는 공공의 안녕질서를 유지하기 위하여 긴급한 조치가 필요하고 국회의 집회를 기다릴 여유가 없을 때에 한하여 최소한으로 필요한 재정·경제상의 처분을 하거나 이에 관하여 법률의 효력을 가지는 명령을 발할 수 있다.\n② 대통령은 국가의 안위에 관계되는 중대한 교전상태에 있어서 국가를 보위하기 위하여 긴급한 조치가 필요하고 국회의 집회가 불가능한 때에 한하여 법률의 효력을 가지는 명령을 발할 수 있다.\n③ 대통령은 제1항과 제2항의 처분 또는 명령을 한 때에는 지체없이 국회에 보고하여 그 승인을 얻어야 한다.\n④ 제3항의 승인을 얻지 못한 때에는 그 처분 또는 명령은 그때부터 효력을 상실한다. 이 경우 그 명령에 의하여 개정 또는 폐지되었던 법률은 그 명령이 승인을 얻지 못한 때부터 당연히 효력을 회복한다.\n⑤ 대통령은 제3항과 제4항의 사유를 지체없이 공포하여야 한다.', np.float64(0.8654384566195557)), ('제80조 대통령은 법률이 정하는 바에 의하여 훈장 기타의 영전을 수여한다.제81조 대통령은 국회에 출석하여 발언하거나 서한으로 의견을 표시할 수 있다.', np.float64(0.8609683513040014))]
[#0] 0.8698584460847605
제72조 대통령은 필요하다고 인정할 때에는 외교·국방·통일 기타 국가안위에 관

In [41]:
def chat_with_user(user_message):
   ai_message = chain.invoke(user_message)
   return ai_message

while True:
   user_message = input("user> ")
   if user_message.lower() == "quit":
      break
   ai_message = chat_with_user(user_message)
   print(f"ai >> {ai_message}")

user> 너는 누구야? 
[('제22조 ① 모든 국민은 학문과 예술의 자유를 가진다.\n② 저작자·발명가·과학기술자와 예술가의 권리는 법률로써 보호한다.', np.float64(0.7826284005489896)), ('제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.\n"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."', np.float64(0.7822210717339839)), ('제10조 모든 국민은 인간으로서의 존엄과 가치를 가지며, 행복을 추구할 권리를 가진다. 국가는 개인이 가지는 불가침의 기본적 인권을 확인하고 이를 보장할 의무를 진다.', np.float64(0.7803799131048039)), ('제1장 총강제1조 ① 대한민국은 민주공화국이다.\n② 대한민국의 주권은 국민에게 있고, 모든 권력은 국민으로부터 나온다.', np.float64(0.7788652893797802))]
[#0] 0.7826284005489896
제22조 ① 모든 국민은 학문과 예술의 자유를 가진다.
② 저작자·발명가·과학기술자와 예술가의 권리는 법률로써 보호한다.
[#1] 0.7822210717339839
제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.
"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."
[#2] 0.7803799131048039
제10조 모든 국민은 인간으로서의 존엄과 가치를 가지며, 행복을 추구할 권리를 가진다. 국가는 개인이 가지는 불가침의 기본적 인권을 확인하고 이를 보장할 의무를 진다.
[#3] 0.7788652893797802
제1장 총강제1조 ① 대한민국은 민주공화국이다.
② 대한민국의 주권은 국민에게 있고, 모든 권력은 국민으로부터 나온다.
ai 

Local Embedding Model

In [ ]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.42.1 which is incompatib

In [ ]:
!pip install -U langchain-huggingface
!pip install -U sentence-transformers

In [ ]:
import langchain
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
raw_documents = SimpleTextLoader('korea_constitution.txt').load()
text_splitter = SimpleCharacterTextSplitter(chunk_size=10, chunk_overlap=0)
documents = text_splitter.split_document(raw_documents)

embed_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-sts")
print(embed_model)


splitting....: 100%|██████████| 152/152 [00:00<00:00, 815261.14it/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.44k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model_name='jhgan/ko-sbert-sts' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [ ]:
db = SimpleVectorStore(documents, embed_model)

embedding...: 100%|██████████| 153/153 [00:39<00:00,  3.92it/s]


In [ ]:
query = "대통령의 임기는?"
docs = db.similarity_search(query)
docs

[('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', np.float64(0.5381745725637745)),
 ('제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.\n② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n④ 법관의 정년은 법률로 정한다.',
  np.float64(0.465767370273822)),
 ('제42조 국회의원의 임기는 4년으로 한다.', np.float64(0.4376108735510348)),
 ('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.',
  np.float64(0.4304336364710237))]

In [ ]:
retriever = db.as_retriever()
chain = SimpleRetrievalQA(retriever)
answer = chain.invoke("대통령의 임기는?")

print(">> ", answer)

[('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', np.float64(0.5381745725637745)), ('제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.\n② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n④ 법관의 정년은 법률로 정한다.', np.float64(0.465767370273822)), ('제42조 국회의원의 임기는 4년으로 한다.', np.float64(0.4376108735510348)), ('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.', np.float64(0.4304336364710237))]
[#0] 0.5381745725637745
제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.
[#1] 0.465767370273822
제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.
② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.
③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.
④ 법관의 정년은 법률로 정한다.
[#2] 0.4376108735510348
제42조 국회의원의 임기는 4년으로 한다.
[#3] 0.4304336364710237
제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.
② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.
>>  대통령의 임기는 5년으로 규정되어 있으며, 중임할 수 없습니다.


In [ ]:
def chat_with_user(user_message):
   ai_message = chain.invoke(user_message)
   return ai_message

while True:
   user_message = input("user> ")
   if user_message.lower() == "quit":
      break
   ai_message = chat_with_user(user_message)
   print(f"ai >> {ai_message}")

user> 대통령의 임기는?
[('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', np.float64(0.5381745725637745)), ('제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.\n② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n④ 법관의 정년은 법률로 정한다.', np.float64(0.465767370273822)), ('제42조 국회의원의 임기는 4년으로 한다.', np.float64(0.4376108735510348)), ('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.', np.float64(0.4304336364710237))]
[#0] 0.5381745725637745
제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.
[#1] 0.465767370273822
제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.
② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.
③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.
④ 법관의 정년은 법률로 정한다.
[#2] 0.4376108735510348
제42조 국회의원의 임기는 4년으로 한다.
[#3] 0.4304336364710237
제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.
② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.
ai >> 대통령의 임기는 5년이며, 중임할 수 없습니다.
user> quit
